# WikiSourceReader Demo

Reading Latin texts from la.wikisource.org `.wiki` files with structured citation preservation.

This demo uses the test fixtures (trimmed excerpts from real WikiSource pages):
- **Prose:** Seneca, *De vita beata* (chapters I–III)
- **Verse:** Vergil, *Aeneid* I (lines 1–30)

In [1]:
from latincyreaders import WikiSourceReader, AnnotationLevel
from pprint import pprint

## Setup

Point the reader at a directory containing `.wiki` files. Here we use the test fixtures.

In [2]:
from pathlib import Path

# Use test fixtures (prose + verse samples)
WIKI_DIR = Path("../tests/fixtures/wikisource")

reader = WikiSourceReader(root=WIKI_DIR, annotation_level=AnnotationLevel.TOKENIZE)
print(f"Root: {reader.root}")
print(f"Annotation level: {reader.annotation_level}")

Root: /Volumes/fiona/work/code/diy/latincy-v3/latincy-readers/tests/fixtures/wikisource
Annotation level: AnnotationLevel.TOKENIZE


## File discovery

The default pattern `**/*.wiki` discovers files recursively. Index pages (TOC-only pages with sub-page links but no content) are automatically skipped during iteration.

In [3]:
fileids = reader.fileids()
print(f"Files found: {len(fileids)}")
pprint(fileids)

Files found: 3
['aeneis/liber_i.wiki', 'de_vita_beata.wiki', 'index_page.wiki']


## Metadata from `{{titulus2}}` templates

WikiSource pages use `{{titulus2|Scriptor=...|OperaeTitulus=...|Annus=...|Genera=...}}` for metadata. The reader extracts this automatically.

In [4]:
# Read the raw wikitext and parse metadata
raw = (WIKI_DIR / "de_vita_beata.wiki").read_text()
meta = reader._parse_titulus(raw)
pprint(meta)

{'author': 'Lucius Annaeus Seneca',
 'date': '-58',
 'genre': 'Philosophia',
 'title': 'De vita beata'}


In [5]:
# Verse file metadata
raw_verse = (WIKI_DIR / "aeneis" / "liber_i.wiki").read_text()
meta_verse = reader._parse_titulus(raw_verse)
pprint(meta_verse)

{'author': 'Publius Vergilius Maro',
 'date': '-19',
 'genre': 'Epos',
 'title': 'Aeneis I'}


## Raw text access (zero NLP overhead)

`texts()` strips all wiki markup and yields clean Latin strings.

In [6]:
for text in reader.texts():
    print(text[:200])
    print("...")
    print()

Arma virumque cano, Troiae qui primus ab oris
Italiam, fato profugus, Laviniaque venit
litora, multum ille et terris iactatus et alto
vi superum saevae memorem Iunonis ob iram;
multa quoque et bello p
...

1. Vivere, Gallio frater, omnes beate volunt, sed ad pervidendum quid sit quod beatam vitam efficiat caligant; adeoque non est facile consequi beatam vitam ut eo quisque ab ea longius recedat quo vehe
...



## Prose: Sections and paragraphs

Prose files use `== I. ==` section headers with numbered paragraphs (`1.`, `2.`, etc.). The reader parses these into `WikiSection` objects and creates citation spans in the format `section.paragraph` (e.g., `I.1`, `III.2`).

In [7]:
# Parse sections from the raw wikitext
sections = reader._parse_sections(raw)

for section in sections:
    print(f"Section {section.header}")
    print(f"  Paragraphs: {len(section.paragraphs)}")
    for para_num, para_text in section.paragraphs:
        print(f"  {section.header}{para_num}: {para_text[:70]}...")
    print()

Section I.
  Paragraphs: 3
  I.1: Vivere, Gallio frater, omnes beate volunt, sed ad pervidendum quid sit...
  I.2: Proponendum est itaque primum quid sit quod appetamus; tunc circumspic...
  I.3: Quamdiu quidem passim vagamur non ducem secuti sed fremitum et clamore...

Section II.
  Paragraphs: 3
  II.1: Decernatur itaque et quo tendimus et qua, non sine perito aliquo cui e...
  II.2: In illis comprehensus aliquis locus et interrogatus incola non patitur...
  II.3: Nihil ergo magis praestandum est quam ne pecorum ritu sequamur anteced...

Section III.
  Paragraphs: 3
  III.1: Atqui nulla res nos maioribus malis implicat quam quod ad rumorem comp...
  III.2: Inde ista tanta coacervatio aliorum super alios ruentium. Quod in stra...
  III.3: Nemo sibi tantummodo errat, sed alieni erroris et causa et auctor est....



In [8]:
# Access prose docs with citation spans
prose_files = [f for f in reader.fileids() if "de_vita_beata" in f]
doc = next(reader.docs(prose_files))

print(f"File: {doc._.fileid}")
print(f"Author: {doc._.metadata.get('author')}")
print(f"Title: {doc._.metadata.get('title')}")
print(f"Content type: {doc._.metadata.get('content_type')}")
print(f"Section spans: {len(doc.spans.get('sections', []))}")
print()

for span in doc.spans["sections"]:
    print(f"{span._.citation}: {span.text[:60]}...")

File: de_vita_beata.wiki
Author: Lucius Annaeus Seneca
Title: De vita beata
Content type: prose
Section spans: 9

I.1: Vivere, Gallio frater, omnes beate volunt, sed ad pervidendu...
I.2: Proponendum est itaque primum quid sit quod appetamus; tunc ...
I.3: Quamdiu quidem passim vagamur non ducem secuti sed fremitum ...
II.1: Decernatur itaque et quo tendimus et qua, non sine perito al...
II.2: In illis comprehensus aliquis locus et interrogatus incola n...
II.3: Nihil ergo magis praestandum est quam ne pecorum ritu sequam...
III.1: Atqui nulla res nos maioribus malis implicat quam quod ad ru...
III.2: Inde ista tanta coacervatio aliorum super alios ruentium. Qu...
III.3: Nemo sibi tantummodo errat, sed alieni erroris et causa et a...


In [9]:
# Iterate sections with the sections() method
for section in reader.sections():
    print(f"{section._.citation}: {section.text[:80]}...")

I.1: Vivere, Gallio frater, omnes beate volunt, sed ad pervidendum quid sit quod beat...
I.2: Proponendum est itaque primum quid sit quod appetamus; tunc circumspiciendum qua...
I.3: Quamdiu quidem passim vagamur non ducem secuti sed fremitum et clamorem dissonum...
II.1: Decernatur itaque et quo tendimus et qua, non sine perito aliquo cui explorata s...
II.2: In illis comprehensus aliquis locus et interrogatus incola non patitur errare; a...
II.3: Nihil ergo magis praestandum est quam ne pecorum ritu sequamur antecedentium gre...
III.1: Atqui nulla res nos maioribus malis implicat quam quod ad rumorem componimur, op...
III.2: Inde ista tanta coacervatio aliorum super alios ruentium. Quod in strage hominum...
III.3: Nemo sibi tantummodo errat, sed alieni erroris et causa et auctor est. Nocet eni...


## Verse: Lines with `{{versus|N}}` citations

Verse files use `<poem>` blocks with `{{versus|N}}` templates for line numbering. The reader parses these into line spans with numeric citations.

In [10]:
# Access verse docs
verse_files = [f for f in reader.fileids() if "liber_i" in f]
verse_doc = next(reader.docs(verse_files))

print(f"File: {verse_doc._.fileid}")
print(f"Author: {verse_doc._.metadata.get('author')}")
print(f"Title: {verse_doc._.metadata.get('title')}")
print(f"Content type: {verse_doc._.metadata.get('content_type')}")
print(f"Line spans: {len(verse_doc.spans.get('lines', []))}")
print()

# First 10 lines with citations
for line in list(verse_doc.spans["lines"])[:10]:
    print(f"  {line._.citation:>3}: {line.text}")

File: aeneis/liber_i.wiki
Author: Publius Vergilius Maro
Title: Aeneis I
Content type: verse
Line spans: 30

    1: Arma virumque cano, Troiae qui primus ab oris
    2: Italiam, fato profugus, Laviniaque venit
    3: litora, multum ille et terris iactatus et alto
    4: vi superum saevae memorem Iunonis ob iram;
    5: multa quoque et bello passus, dum conderet urbem,
    6: inferretque deos Latio, genus unde Latinum,
    7: Albanique patres, atque altae moenia Romae.
    8: Musa, mihi causas memora, quo numine laeso,
    9: quidve dolens, regina deum tot volvere casus
   10: insignem pietate virum, tot adire labores


In [11]:
# Iterate verse lines with the lines() method
for line in reader.lines():
    print(f"  {line._.citation:>3}: {line.text}")

    1: Arma virumque cano, Troiae qui primus ab oris
    2: Italiam, fato profugus, Laviniaque venit
    3: litora, multum ille et terris iactatus et alto
    4: vi superum saevae memorem Iunonis ob iram;
    5: multa quoque et bello passus, dum conderet urbem,
    6: inferretque deos Latio, genus unde Latinum,
    7: Albanique patres, atque altae moenia Romae.
    8: Musa, mihi causas memora, quo numine laeso,
    9: quidve dolens, regina deum tot volvere casus
   10: insignem pietate virum, tot adire labores
   11: impulerit. Tantaene animis caelestibus irae?
   12: Urbs antiqua fuit, Tyrii tenuere coloni,
   13: Karthago, Italiam contra Tiberinaque longe
   14: ostia, dives opum studiisque asperrima belli;
   15: quam Iuno fertur terris magis omnibus unam
   16: posthabita coluisse Samo; hic illius arma,
   17: hic currus fuit; hoc regnum dea gentibus esse,
   18: si qua fata sinant, iam tum tenditque fovetque.
   19: Progeniem sed enim Troiano a sanguine duci
   20: audierat, Tyria

## NLP annotation

Switch to `BASIC` or `FULL` annotation for lemmatization, POS tagging, and morphological analysis.

In [12]:
reader_nlp = WikiSourceReader(
    root=WIKI_DIR,
    annotation_level=AnnotationLevel.BASIC,
)

# Lemmatized tokens from De vita beata
prose_doc = next(reader_nlp.docs(prose_files))

print("First 20 tokens with lemmas and POS:")
print()
for token in prose_doc[:20]:
    if not token.is_punct:
        print(f"  {token.text:<20} {token.lemma_:<20} {token.pos_}")

First 20 tokens with lemmas and POS:

  1                    1                    NUM
  Vivere               uiuo                 VERB
  Gallio               Gallio               PROPN
  frater               frater               NOUN
  omnes                omnis                DET
  beate                beatus               ADV
  volunt               uolo                 VERB
  sed                  sed                  CCONJ
  ad                   ad                   ADP
  pervidendum          peruideo             VERB
  quid                 quis                 PRON
  sit                  sum                  AUX
  quod                 quod                 SCONJ
  beatam               beatus               ADJ
  vitam                uita                 NOUN
  efficiat             efficio              VERB


In [13]:
# Sentence segmentation
print(f"Sentences in De vita beata (I-III):")
print()
for i, sent in enumerate(prose_doc.sents):
    print(f"  Sent {i+1}: {sent.text[:80]}...")
    if i >= 4:
        print(f"  ... ({len(list(prose_doc.sents))} sentences total)")
        break

Sentences in De vita beata (I-III):

  Sent 1: 1....
  Sent 2: Vivere, Gallio frater, omnes beate volunt, sed ad pervidendum quid sit quod beat...
  Sent 3: adeoque non est facile consequi beatam vitam ut eo quisque ab ea longius recedat...
  Sent 4: quae ubi in contrarium ducit, ipsa velocitas maioris intervalli causa fit. 2....
  Sent 5: Proponendum est itaque primum quid sit quod appetamus;...
  ... (17 sentences total)


In [14]:
# Lemmatized verse lines
verse_doc_nlp = next(reader_nlp.docs(verse_files))

print("Aeneid I.1-5 with lemmas:")
print()
for line in list(verse_doc_nlp.spans["lines"])[:5]:
    lemmas = " ".join(t.lemma_ for t in line if not t.is_punct and not t.is_space)
    print(f"  {line._.citation:>3}: {line.text}")
    print(f"       → {lemmas}")
    print()

Aeneid I.1-5 with lemmas:

    1: Arma virumque cano, Troiae qui primus ab oris
       → arma uir que cano Troia qui primus ab ora

    2: Italiam, fato profugus, Laviniaque venit
       → Italia fatum profugus Lauinia que uenio

    3: litora, multum ille et terris iactatus et alto
       → litus multum ille et terra iactatus et altus

    4: vi superum saevae memorem Iunonis ob iram;
       → uis superus saevus memor Iuno ob ira

    5: multa quoque et bello passus, dum conderet urbem,
       → multus quoque et bellum patior dum condo urbs



## Downloading from la.wikisource.org

The `download()` classmethod fetches pages via the MediaWiki API. It saves raw wikitext as `.wiki` files and can recursively follow sub-page links.

```python
# Download a prose text
paths = WikiSourceReader.download("De_vita_beata", "~/latincy_data/wikisource")

# Download a multi-book work (follows sub-pages automatically)
paths = WikiSourceReader.download("Aeneis", "~/latincy_data/wikisource")

# Then create a reader pointing at the download directory
reader = WikiSourceReader("~/latincy_data/wikisource")
```

## Summary

| Feature | Prose | Verse |
|---------|-------|-------|
| File format | `.wiki` (wikitext) | `.wiki` (wikitext) |
| Content markers | `== I. ==` sections, `1.` paragraphs | `<poem>`, `{{versus\|N}}` |
| Citation format | `I.1`, `II.3` | `1`, `15`, `30` |
| Span group | `doc.spans["sections"]` | `doc.spans["lines"]` |
| Iterator method | `reader.sections()` | `reader.lines()` |